In [2]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

# 1. Localizar el parquet limpio
RAIZ = next(
    p
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "proyecto_equipo" / "datos").is_dir() or (p / "datos").is_dir()
)
RUTA = next(RAIZ.glob("**/ai_company_adoption_limpio.parquet"))
df = pd.read_parquet(RUTA)

# Partición agrupada por empresa (80% train / 20% test, semilla 2026)
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=2026)
train_idx, test_idx = next(gss.split(df, groups=df["company_id"]))
df_train = df.iloc[train_idx].copy()

# ==============================================================================
# A. MODELO 1: REGRESIÓN RIDGE (Predicción de ai_adoption_rate)
# ==============================================================================
cols_log = ["num_employees", "annual_revenue_usd_millions"]
cols_esc = ["company_age", "survey_year"]
cols_cat = ["country", "industry"]
cols_reg = cols_log + cols_esc + cols_cat

X_reg = df_train[cols_reg].copy()
y_reg = df_train["ai_adoption_rate"].astype(float).values

# Calculamos medias y desvíos exactos para no depender de sklearn en la app
X_log_vals = np.log1p(X_reg[cols_log].astype(float).values)
log_mean = X_log_vals.mean(axis=0)
log_scale = X_log_vals.std(axis=0)

X_esc_vals = X_reg[cols_esc].astype(float).values
esc_mean = X_esc_vals.mean(axis=0)
esc_scale = X_esc_vals.std(axis=0)

paises = sorted(df["country"].astype(str).unique().tolist())
industrias = sorted(df["industry"].astype(str).unique().tolist())


def codificar_reg(df_in):
  n = len(df_in)
  x_l = (np.log1p(df_in[cols_log].astype(float).values) - log_mean) / log_scale
  x_e = (df_in[cols_esc].astype(float).values - esc_mean) / esc_scale
  c_mat = np.zeros((n, len(paises)))
  i_mat = np.zeros((n, len(industrias)))
  for idx, val in enumerate(df_in["country"].astype(str)):
    if val in paises:
      c_mat[idx, paises.index(val)] = 1.0
  for idx, val in enumerate(df_in["industry"].astype(str)):
    if val in industrias:
      i_mat[idx, industrias.index(val)] = 1.0
  return np.hstack([x_l, x_e, c_mat, i_mat])


X_reg_mat = codificar_reg(X_reg)
ridge = Ridge(alpha=200.0, random_state=2026)
ridge.fit(X_reg_mat, y_reg)

nombres_features_reg = (
    cols_log
    + cols_esc
    + [f"country_{c}" for c in paises]
    + [f"industry_{i}" for i in industrias]
)

# ==============================================================================
# B. MODELO 2: CLASIFICACIÓN LOGÍSTICA BALANCEADA (ai_adoption_stage)
# ==============================================================================
cols_num_clf = [
    "num_employees",
    "annual_revenue_usd_millions",
    "company_age",
    "survey_year",
    "ai_training_hours",
    "num_ai_tools_used",
]
cols_log_clf = ["num_employees", "annual_revenue_usd_millions", "ai_training_hours"]
cols_lin_clf = ["company_age", "survey_year", "num_ai_tools_used"]

X_clf_log = np.log1p(df_train[cols_log_clf].astype(float).values)
clf_log_mean = X_clf_log.mean(axis=0)
clf_log_scale = X_clf_log.std(axis=0)

X_clf_lin = df_train[cols_lin_clf].astype(float).values
clf_lin_mean = X_clf_lin.mean(axis=0)
clf_lin_scale = X_clf_lin.std(axis=0)


def codificar_clf(df_in):
  n = len(df_in)
  x_l = (
      np.log1p(df_in[cols_log_clf].astype(float).values) - clf_log_mean
  ) / clf_log_scale
  x_e = (
      df_in[cols_lin_clf].astype(float).values - clf_lin_mean
  ) / clf_lin_scale
  c_mat = np.zeros((n, len(paises)))
  i_mat = np.zeros((n, len(industrias)))
  for idx, val in enumerate(df_in["country"].astype(str)):
    if val in paises:
      c_mat[idx, paises.index(val)] = 1.0
  for idx, val in enumerate(df_in["industry"].astype(str)):
    if val in industrias:
      i_mat[idx, industrias.index(val)] = 1.0
  return np.hstack([x_l, x_e, c_mat, i_mat])


X_clf_mat = codificar_clf(df_train)
clases_orden = ["none", "pilot", "partial", "full"]
y_clf = df_train["ai_adoption_stage"].astype(str).values

logreg = LogisticRegression(
    C=10.0, class_weight="balanced", max_iter=1000, random_state=2026
)
logreg.fit(X_clf_mat, y_clf)

# ==============================================================================
# C. MODELO 3: CLUSTERING K-MEANS (Segmentación de Empresas por Impacto)
# ==============================================================================
cols_impacto = [
    "task_automation_rate",
    "time_saved_per_week",
    "productivity_change_percent",
    "revenue_growth_percent",
]
df_emp = df.groupby("company_id")[cols_impacto].mean()
df_emp["desplazados_x1000"] = (
    df.groupby("company_id")["jobs_displaced"].sum()
    / df.groupby("company_id")["num_employees"].first()
) * 1000
df_emp["creados_x1000"] = (
    df.groupby("company_id")["jobs_created"].sum()
    / df.groupby("company_id")["num_employees"].first()
) * 1000
cols_cluster = df_emp.columns.tolist()

vals_cluster = df_emp[cols_cluster].values
clus_mean = vals_cluster.mean(axis=0)
clus_scale = vals_cluster.std(axis=0)
vals_cluster_esc = (vals_cluster - clus_mean) / clus_scale

kmeans = KMeans(n_clusters=3, random_state=2026, n_init=10)
df_emp["cluster"] = kmeans.fit_predict(vals_cluster_esc)

# Ordenar nombres según automatización promedio para que siempre coincidan
orden_clusters = (
    df_emp.groupby("cluster")["task_automation_rate"]
    .mean()
    .sort_values(ascending=False)
    .index.tolist()
)
nombres_clusters = {
    orden_clusters[0]: "Líderes Integrales",
    orden_clusters[1]: "Automatizadoras de Coste",
    orden_clusters[2]: "En Transición / Rezagadas",
}
perfiles_cluster = df_emp.groupby("cluster")[cols_cluster].mean().round(2)

# Muestra de 400 empresas para graficar la nube de puntos en la app sin pesar
muestra_emp = df_emp.sample(400, random_state=2026).copy()
muestra_emp["perfil"] = muestra_emp["cluster"].map(nombres_clusters)

# ==============================================================================
# D. GUARDAR ARTEFACTO ULTRALIGERO (SOLO NUMPY Y PANDAS, CERO SCIPY)
# ==============================================================================
artefacto = {
    "paises": paises,
    "industrias": industrias,
    "regresion": {
        "cols_log": cols_log,
        "cols_esc": cols_esc,
        "log_mean": log_mean,
        "log_scale": log_scale,
        "esc_mean": esc_mean,
        "esc_scale": esc_scale,
        "coef": ridge.coef_,
        "intercept": float(ridge.intercept_),
        "feature_names": nombres_features_reg,
        "media_historica": float(y_reg.mean()),
        "p25": float(np.percentile(y_reg, 25)),
        "p50": float(np.percentile(y_reg, 50)),
        "p75": float(np.percentile(y_reg, 75)),
    },
    "clasificacion": {
        "cols_log": cols_log_clf,
        "cols_lin": cols_lin_clf,
        "log_mean": clf_log_mean,
        "log_scale": clf_log_scale,
        "lin_mean": clf_lin_mean,
        "lin_scale": clf_lin_scale,
        "clases": logreg.classes_.tolist(),
        "coef": logreg.coef_,
        "intercept": logreg.intercept_,
        "importancia_perm": {
            "Horas de capacitación (ai_training_hours)": 0.180,
            "Herramientas de IA (num_ai_tools_used)": 0.093,
            "Industria (industry)": 0.008,
            "Empleados (num_employees)": 0.004,
            "País (country)": 0.003,
            "Ingresos anuales (annual_revenue)": 0.002,
        },
    },
    "clustering": {
        "columnas": cols_cluster,
        "medias": clus_mean,
        "escalas": clus_scale,
        "centroides": kmeans.cluster_centers_,
        "nombres": nombres_clusters,
        "perfiles": perfiles_cluster,
        "muestra": muestra_emp,
    },
}

carpeta_destino = (
    RAIZ / "proyecto_equipo" if (RAIZ / "proyecto_equipo").is_dir() else RAIZ
)
ruta_salida = carpeta_destino / "modelos_tp4.joblib"
joblib.dump(artefacto, ruta_salida)
print(f"Artefacto guardado con éxito en: {ruta_salida}")

Artefacto guardado con éxito en: C:\AAG\Practica\proyecto_equipo\modelos_tp4.joblib
